# 🚀 OpenSource Clipping — Quick Start Notebook

> **⚡ Quick Start**: If you just want to get up and running as fast as possible,
> simply run all cells from top to bottom. The only thing you need to change is
> the **YouTube URL** and your **API Key** in the Configuration section below.

---

## What is this?

This notebook is a **minimal, beginner-friendly** implementation of
[OpenSource Clipping](https://github.com/NaufalRizqullah/opensource-clipping) —
an AI-powered tool that automatically transforms long-form videos into
cinematic short-form highlight clips with:

- 🎯 AI-curated highlight moments (powered by Google Gemini)
- 🎙️ Word-level transcription (powered by Faster-Whisper)
- 📱 Smart face-tracking auto-framing for vertical (9:16) format
- 💬 Karaoke-style animated subtitles
- 🎬 Cinematic hook teaser intro with glitch transition

## How to use this notebook

| Step | Section | What to do |
|------|---------|------------|
| 1 | **Setup** | Run the setup cells to clone the repo & install dependencies |
| 2 | **Configuration** | Set your YouTube URL, API key, and clip preferences |
| 3 | **Run** | Execute the pipeline — sit back and wait for your clips! |
| 4 | **Download** | Download the generated clips from the `outputs/` folder |

### ⚠️ Requirements

- **Runtime**: Set your Colab runtime to **T4 GPU** (`Runtime > Change runtime type > T4 GPU`)
- **API Key**: You need a free [Google Gemini API Key](https://aistudio.google.com/apikey)

---

## 1. Setup — Clone Repository & Install Dependencies

This section clones the OpenSource Clipping repository and installs all
required Python packages. It also installs **FFmpeg** (needed for video
processing) and **Deno** (used internally by some modules).

> 💡 **Tip**: This cell only needs to run once per Colab session.

In [1]:
# ──────────────────────────────────────────────────────────────────────
# 1a. Clone the repository into the current working directory
# ──────────────────────────────────────────────────────────────────────
!rm -rf ./* ./.*
!git clone https://github.com/NaufalRizqullah/opensource-clipping.git .

Cloning into '.'...
remote: Enumerating objects: 1905, done.
remote: Counting objects: 100% (1035/1035), done.
remote: Compressing objects: 100% (234/234), done.
remote: Total 1905 (delta 882), reused 817 (delta 801), pack-reused 870 (from 1)
Receiving objects: 100% (1905/1905), 75.20 MiB | 38.10 MiB/s, done.
Resolving deltas: 100% (1255/1255), done.


In [3]:
%%capture
# ──────────────────────────────────────────────────────────────────────
# 1b. Install system dependencies (FFmpeg + Deno) and Python packages
# ──────────────────────────────────────────────────────────────────────
import os

# FFmpeg — required for video encoding/decoding
!apt-get -qq update
!apt-get -qq install -y ffmpeg

# Deno — used by internal subtitle rendering modules
!curl -fsSL https://deno.land/install.sh | sh
os.environ["PATH"] += ":/root/.deno/bin"

# Python dependencies
!pip install -r requirements.txt

In [5]:

import sys
import os
import shutil
import subprocess
from pathlib import Path

print("========== PYTHON ==========")
print(sys.version)

print("\n========== GPU ==========")
try:
    subprocess.run(["nvidia-smi"], check=True)
except Exception as e:
    print("GPU not available:", e)

print("\n========== PYTORCH ==========")
try:
    import torch
    print("Version:", torch.__version__)
    print("CUDA:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
except Exception as e:
    print(e)

print("\n========== FFMPEG ==========")
print("FFmpeg:", shutil.which("ffmpeg"))
if shutil.which("ffmpeg"):
    subprocess.run(["ffmpeg", "-version"])

print("\n========== INTERNET ==========")
try:
    import urllib.request
    urllib.request.urlopen("https://github.com", timeout=10)
    print("Internet: Connected")
except Exception as e:
    print("Internet:", e)

print("\n========== EXISTING FILES ==========")
for p in [
    "/content/clipping",
    "/content/opensource-clipping",
    "/content/.venv312"
]:
    print(p, "EXISTS" if os.path.exists(p) else "NOT FOUND")

print("\n========== DISK SPACE ==========")
print(shutil.disk_usage("/content"))


========== PYTHON ==========
3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]

========== GPU ==========

========== PYTORCH ==========
Version: 2.14.1+cu130
CUDA: True
GPU: Tesla T4

========== FFMPEG ==========
FFmpeg: /usr/bin/ffmpeg

========== INTERNET ==========
Internet: Connected

========== EXISTING FILES ==========
/content/clipping EXISTS
/content/opensource-clipping NOT FOUND
/content/.venv312 NOT FOUND

========== DISK SPACE ==========
usage(total=120942624768, used=1663758336, free=119262089216)


In [6]:

from pathlib import Path
import subprocess

REPO = Path("/content/clipping")

if not REPO.exists():
    print("ERROR: Existing project folder not found.")
else:
    print("========== GIT INFORMATION ==========")

    def git_info(*args):
        result = subprocess.run(
            ["git", "-C", str(REPO), *args],
            capture_output=True, text=True
        )
        return result.stdout.strip() or result.stderr.strip()

    print("Git repository:", (REPO / ".git").exists())

    if (REPO / ".git").exists():
        print("Remote:", git_info("remote", "get-url", "origin"))
        print("Branch:", git_info("branch", "--show-current"))
        print("Commit:", git_info("rev-parse", "--short", "HEAD"))
        print("Changes:")
        print(git_info("status", "--short"))

    print("\n========== IMPORTANT FILES ==========")

    files = [
        "README.md",
        "AGENTS.md",
        "requirements.txt",
        "requirements-kaggle.txt",
        "constraints-kaggle.txt",
        "pyproject.toml",
        ".env.sample",
        "main.py",
        "clipping/config.py",
        "clipping/engine.py",
        "scripts/kaggle_studio.py",
        "notebooks/Quick_Start.ipynb",
        "notebooks/kaggle-studio-server.ipynb"
    ]

    for name in files:
        p = REPO / name
        print(name, "FOUND" if p.exists() else "MISSING")

    print("\n========== PROJECT STRUCTURE ==========")

    for p in sorted(REPO.iterdir()):
        if p.name not in [".git", ".cache", "__pycache__"]:
            print(("[DIR] " if p.is_dir() else "[FILE] ") + p.name)


========== GIT INFORMATION ==========
Git repository: False

========== IMPORTANT FILES ==========
README.md MISSING
AGENTS.md MISSING
requirements.txt MISSING
requirements-kaggle.txt MISSING
constraints-kaggle.txt MISSING
pyproject.toml MISSING
.env.sample MISSING
main.py MISSING
clipping/config.py MISSING
clipping/engine.py MISSING
scripts/kaggle_studio.py MISSING
notebooks/Quick_Start.ipynb MISSING
notebooks/kaggle-studio-server.ipynb MISSING

========== PROJECT STRUCTURE ==========
[FILE] __init__.py
[FILE] config.py
[FILE] diarization.py
[FILE] engine.py
[FILE] hook_manager.py
[FILE] metadata.py
[DIR] providers
[FILE] runner.py
[DIR] studio
[FILE] studio.py
[FILE] voiceover.py


In [7]:

from pathlib import Path

ROOT = Path("/content")

print("========== CONTENT ROOT ==========")

for p in sorted(ROOT.iterdir()):
    if p.name not in [".config", ".cache", "__pycache__"]:
        print(("[DIR] " if p.is_dir() else "[FILE] ") + p.name)

print("\n========== SEARCH PROJECT FILES ==========")

targets = [
    "README.md",
    "requirements.txt",
    "main.py",
    "AGENTS.md",
    "Quick_Start.ipynb"
]

for target in targets:
    matches = list(ROOT.glob(f"**/{target}"))
    matches = [
        p for p in matches
        if ".cache" not in p.parts
        and ".git" not in p.parts
    ]
    print(f"\n{target}:")
    for p in matches[:10]:
        print(" ", p)
    if not matches:
        print("  NOT FOUND")


========== CONTENT ROOT ==========
[FILE] .dockerignore
[FILE] .env.sample
[DIR] .git
[DIR] .github
[FILE] .gitignore
[FILE] .python-version
[FILE] AGENTS.md
[FILE] CHANGELOG.md
[FILE] Dockerfile
[FILE] LICENSE
[FILE] README.md
[FILE] README_ID.md
[DIR] assets
[FILE] cleanup.sh
[DIR] clipping
[FILE] constraints-kaggle.txt
[FILE] docker-compose.yml
[DIR] docs
[DIR] facebook_uploader
[FILE] main.py
[DIR] notebooks
[FILE] pyproject.toml
[FILE] pytest.ini
[FILE] requirements-kaggle.txt
[FILE] requirements-test.txt
[FILE] requirements-vllm-kaggle.txt
[FILE] requirements.txt
[FILE] run_fb_upload.py
[FILE] run_upload.py
[DIR] scripts
[DIR] tests
[FILE] upload_safety.json
[DIR] web
[DIR] wiki
[DIR] youtube_tracker
[DIR] youtube_uploader

========== SEARCH PROJECT FILES ==========

README.md:
  /content/README.md
  /content/youtube_tracker/README.md
  /content/assets/bgm/README.md

requirements.txt:
  /content/requirements.txt

main.py:
  /content/main.py

AGENTS.md:
  /content/AGENTS.md

Quick

In [8]:

from pathlib import Path
import subprocess

ROOT = Path("/content")

def read_file(name, limit=18000):
    p = ROOT / name
    print(f"\n{'='*65}\n{name}\n{'='*65}")
    if not p.exists():
        print("NOT FOUND")
        return
    content = p.read_text(errors="replace")
    print(content[:limit])
    if len(content) > limit:
        print("\n... OUTPUT TRUNCATED ...")

# 1. Git information
print("========== GIT ==========")
def git(*args):
    r = subprocess.run(
        ["git", "-C", str(ROOT), *args],
        capture_output=True, text=True
    )
    return r.stdout.strip() or r.stderr.strip()

print("Remote:\n", git("remote", "-v"))
print("Branch:", git("branch", "--show-current"))
print("Commit:", git("rev-parse", "HEAD"))
print("Working tree:\n", git("status", "--short"))

# 2. Instructions and documentation
read_file("AGENTS.md")
read_file("README.md", 22000)

# 3. Dependency configurations
read_file("requirements.txt")
read_file("requirements-kaggle.txt")
read_file("constraints-kaggle.txt")
read_file("requirements-vllm-kaggle.txt")
read_file("pyproject.toml", 12000)

# 4. Environment variable names only
print("\n========== ENVIRONMENT TEMPLATE ==========")
env_file = ROOT / ".env.sample"
if env_file.exists():
    for line in env_file.read_text(errors="replace").splitlines():
        if "=" in line and not line.strip().startswith("#"):
            key = line.split("=", 1)[0].strip()
            print(key + "=<VALUE HIDDEN>")
        else:
            print(line)

# 5. Setup scripts and notebooks
print("\n========== SETUP FILES ==========")
for folder in ["scripts", "notebooks"]:
    p = ROOT / folder
    if p.exists():
        print(f"\n{folder}/")
        for item in sorted(p.iterdir()):
            print(item.name)


========== GIT ==========
Remote:
 origin	https://github.com/NaufalRizqullah/opensource-clipping.git (fetch)
origin	https://github.com/NaufalRizqullah/opensource-clipping.git (push)
Branch: main
Commit: b705c9eac5db2f40aec69662da80aae2d41fd87f
Working tree:
 

AGENTS.md
# Project Instructions

## Knowledge Sources

Before implementing changes, consult:

- README.md for application overview.
- CHANGELOG.md for historical changes.
- Wiki for functional and business requirements.
- docs/architecture.md for system architecture.
- docs/modules.md for module responsibilities.

## Rules

- Follow existing architecture and conventions.
- Reuse existing services before adding new code.
- Check relevant changelog entries before changing
  existing behavior.
- Preserve backward compatibility unless approved.
- Update documentation when behavior changes.
- Add or update tests for affected features.

## Workflow

1. Identify the affected module.
2. Read relevant documentation.
3. Inspect its curren

In [9]:

from pathlib import Path
import subprocess

ROOT = Path("/content")

def read_file(name, limit=18000):
    p = ROOT / name
    print(f"\n{'='*65}\n{name}\n{'='*65}")
    if not p.exists():
        print("NOT FOUND")
        return
    content = p.read_text(errors="replace")
    print(content[:limit])
    if len(content) > limit:
        print("\n... OUTPUT TRUNCATED ...")

# 1. Git information
print("========== GIT ==========")
def git(*args):
    r = subprocess.run(
        ["git", "-C", str(ROOT), *args],
        capture_output=True, text=True
    )
    return r.stdout.strip() or r.stderr.strip()

print("Remote:\n", git("remote", "-v"))
print("Branch:", git("branch", "--show-current"))
print("Commit:", git("rev-parse", "HEAD"))
print("Working tree:\n", git("status", "--short"))

# 2. Instructions and documentation
read_file("AGENTS.md")
read_file("README.md", 22000)

# 3. Dependency configurations
read_file("requirements.txt")
read_file("requirements-kaggle.txt")
read_file("constraints-kaggle.txt")
read_file("requirements-vllm-kaggle.txt")
read_file("pyproject.toml", 12000)

# 4. Environment variable names only
print("\n========== ENVIRONMENT TEMPLATE ==========")
env_file = ROOT / ".env.sample"
if env_file.exists():
    for line in env_file.read_text(errors="replace").splitlines():
        if "=" in line and not line.strip().startswith("#"):
            key = line.split("=", 1)[0].strip()
            print(key + "=<VALUE HIDDEN>")
        else:
            print(line)

# 5. Setup scripts and notebooks
print("\n========== SETUP FILES ==========")
for folder in ["scripts", "notebooks"]:
    p = ROOT / folder
    if p.exists():
        print(f"\n{folder}/")
        for item in sorted(p.iterdir()):
            print(item.name)


========== GIT ==========
Remote:
 origin	https://github.com/NaufalRizqullah/opensource-clipping.git (fetch)
origin	https://github.com/NaufalRizqullah/opensource-clipping.git (push)
Branch: main
Commit: b705c9eac5db2f40aec69662da80aae2d41fd87f
Working tree:
 

AGENTS.md
# Project Instructions

## Knowledge Sources

Before implementing changes, consult:

- README.md for application overview.
- CHANGELOG.md for historical changes.
- Wiki for functional and business requirements.
- docs/architecture.md for system architecture.
- docs/modules.md for module responsibilities.

## Rules

- Follow existing architecture and conventions.
- Reuse existing services before adding new code.
- Check relevant changelog entries before changing
  existing behavior.
- Preserve backward compatibility unless approved.
- Update documentation when behavior changes.
- Add or update tests for affected features.

## Workflow

1. Identify the affected module.
2. Read relevant documentation.
3. Inspect its curren

In [10]:

from pathlib import Path

ROOT = Path("/content")

files = [
    "CHANGELOG.md",
    "docs/architecture.md",
    "docs/modules.md",
    "docs/KAGGLE_STUDIO.md",
    "docs/providers.md",
    "docs/KAGGLE_VLLM.md",
    "scripts/kaggle_python.py",
    "scripts/kaggle_studio.py",
    "scripts/studio_server.py",
]

for name in files:
    path = ROOT / name
    print("\n" + "=" * 65)
    print("FILE:", name)
    print("=" * 65)

    if not path.exists():
        print("NOT FOUND")
        continue

    content = path.read_text(errors="replace")

    # Recent changelog only
    if name == "CHANGELOG.md":
        content = content[:10000]

    print(content[:14000])

    if len(content) > 14000:
        print("\n... TRUNCATED ...")

print("\n========== NOTEBOOK FILES ==========")
for p in sorted((ROOT / "notebooks").glob("*.ipynb")):
    print(p.name, f"({p.stat().st_size // 1024} KB)")



FILE: CHANGELOG.md
# Changelog

All notable changes to the **OpenSource Clipping** project will be documented in this file.

**The Standard Structure (SemVer)**
- **Major (X.y.z)**: Incremented for incompatible API changes (breaking changes).
- **Minor (x.Y.z)**: Incremented for new functionality introduced in a backward-compatible manner.
- **Patch (x.y.Z)**: Incremented for backward-compatible bug fixes or minor patches.


## [v1.16.4]

### Added
- **GitHub Pages Studio languages**: English (`en`, default) and Indonesian (`id`) across the Dashboard, New Job, Job Detail, Settings, and connection dialogs. The sidebar language selector remembers an explicit browser preference and updates labels, validation, statuses, and dates immediately while preserving form values, progress subscriptions, and video elements. Backend payloads, generated content, and raw server diagnostics retain their existing behavior. Added localization regression tests and usage/contributor documentation.

## [v1.

## 2. Configuration — Set Your Preferences

This is the **only section you need to edit**. Configure the following:

| Parameter | Description | Default |
|-----------|-------------|---------|
| `YOUTUBE_URL` | The YouTube video URL you want to clip | *(required)* |
| `TOTAL_CLIPS` | How many highlight clips to generate | `3` |
| `WHISPER_MODEL` | Whisper model size for transcription accuracy | `large-v3` |
| `WHISPER_DEVICE` | Device for Whisper inference (`cuda` or `cpu`) | `cuda` |
| `WHISPER_COMPUTE_TYPE` | Compute precision (`float16` for Colab T4, `float32` for CPU) | `float16` |

### Whisper Model Options

| Model | Speed | Accuracy | VRAM |
|-------|-------|----------|------|
| `tiny` | ⚡⚡⚡⚡⚡ | ★☆☆☆☆ | ~1 GB |
| `base` | ⚡⚡⚡⚡ | ★★☆☆☆ | ~1 GB |
| `small` | ⚡⚡⚡ | ★★★☆☆ | ~2 GB |
| `medium` | ⚡⚡ | ★★★★☆ | ~5 GB |
| `large-v3` | ⚡ | ★★★★★ | ~10 GB |

> 💡 **Recommendation**: Use `large-v3` with `float16` on Colab T4 GPU for best
> results. If you're on a free Kaggle notebook or CPU-only environment, use
> `small` or `medium` with `float32`.

In [11]:

from pathlib import Path
import sys
import subprocess

ROOT = Path("/content")

# Verify actual project root
assert (ROOT / "scripts/kaggle_python.py").is_file()
assert (ROOT / "scripts/kaggle_studio.py").is_file()
assert (ROOT / "requirements-kaggle.txt").is_file()

import os
os.chdir(ROOT)

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Official repository Python bootstrap
from scripts.kaggle_python import (
    bootstrap_environment,
    ensure_python
)

PYTHON_CACHE = ROOT / ".cache" / "kaggle-python"

print("Preparing isolated Python 3.12...")

BOOTSTRAP_PYTHON = ensure_python(PYTHON_CACHE)

print("\n========== VERIFICATION ==========")
print("Colab Kernel:", sys.version)

result = subprocess.run(
    [str(BOOTSTRAP_PYTHON), "--version"],
    cwd=ROOT,
    env=bootstrap_environment(PYTHON_CACHE),
    capture_output=True,
    text=True,
    check=True
)

print("Application Python:", result.stdout.strip())
print("Python executable:", BOOTSTRAP_PYTHON)
print("\nPython preparation completed.")


Preparing isolated Python 3.12...

========== VERIFICATION ==========
Colab Kernel: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
Application Python: Python 3.12.3
Python executable: /usr/bin/python3.12

Python preparation completed.


In [ ]:
from pathlib import Path
import subprocess
import os

# Google Colab project directory
ROOT = Path("/content")

# Python 3.12 prepared in previous step
PYTHON = Path("/usr/bin/python3.12")

# Project-specific virtual environment
VENV = ROOT / ".cache" / "colab-clipping" / "venv"

# Verify project files
assert (ROOT / "main.py").exists(), "main.py missing"
assert (ROOT / "requirements-kaggle.txt").exists()
assert (ROOT / "scripts/kaggle_studio.py").exists()
assert PYTHON.exists(), "Python 3.12 not found"

os.chdir(ROOT)

print("Platform: Google Colab")
print("Project:", ROOT)
print("Python:", PYTHON)
print("Environment:", VENV)

# Run repository's official environment setup
subprocess.run(
    [
        str(PYTHON),
        str(ROOT / "scripts/kaggle_studio.py"),
        "setup",
        "--venv",
        str(VENV)
    ],
    cwd=ROOT,
    check=True
)

# Verify environment
APP_PYTHON = VENV / "bin" / "python"

subprocess.run(
    [str(APP_PYTHON), "--version"],
    check=True
)

print("\nColab project environment setup completed.")

Platform: Google Colab
Project: /content
Python: /usr/bin/python3.12
Environment: /content/.cache/colab-clipping/venv


In [ ]:
# ══════════════════════════════════════════════════════════════════════
#  CONFIGURATION — Edit the values below to match your preferences
# ══════════════════════════════════════════════════════════════════════

# ── YouTube URL ──────────────────────────────────────────────────────
# Paste the YouTube video URL you want to generate clips from.
YOUTUBE_URL = "https://www.youtube.com/watch?v=YOUR_VIDEO_ID"  # <-- CHANGE THIS

# ── Total Clips ─────────────────────────────────────────────────────
# Number of highlight clips to generate from the video.
# More clips = longer processing time. Start small (1-3) for testing.
TOTAL_CLIPS = 3

# ── Whisper Configuration ────────────────────────────────────────────
# These control the speech-to-text transcription step.
WHISPER_MODEL = "large-v3"        # Model size (see table above)
WHISPER_DEVICE = "cuda"            # "cuda" for GPU, "cpu" for CPU-only
WHISPER_COMPUTE_TYPE = "float16"   # "float16" for Colab T4, "float32" for CPU/Kaggle

# ══════════════════════════════════════════════════════════════════════
#  OPTIONAL — Advanced settings (safe to leave as defaults)
# ══════════════════════════════════════════════════════════════════════

ASPECT_RATIO = "9:16"              # Output aspect ratio: "9:16", "16:9", "1:1"
FONT_STYLE = "HORMOZI"             # Subtitle style: "DEFAULT", "HORMOZI", "CINEMATIC", "STORYTELLER"
HOOK_DURATION = 3                  # Hook teaser duration in seconds
WORDS_PER_SUBTITLE = 5             # Max words per subtitle group

print("✅ Configuration loaded!")
print(f"   URL           : {YOUTUBE_URL}")
print(f"   Total Clips   : {TOTAL_CLIPS}")
print(f"   Whisper Model : {WHISPER_MODEL} ({WHISPER_DEVICE}, {WHISPER_COMPUTE_TYPE})")
print(f"   Aspect Ratio  : {ASPECT_RATIO}")
print(f"   Font Style    : {FONT_STYLE}")

## 3. API Key Setup

Your **Google Gemini API Key** is required for the AI analysis step.
The recommended way is to store it in **Colab Secrets**:

1. Click the 🔑 **Key icon** in the left sidebar of Colab
2. Add a new secret named `GOOGLE_API_KEY`
3. Paste your API key as the value
4. Toggle the **"Notebook access"** switch ON

> 🔗 Get a free API key at: https://aistudio.google.com/apikey

**Optional keys** (for extra features):
- `PEXELS_API_KEY` — Enables automatic B-roll stock footage ([get one here](https://www.pexels.com/api/))
- `HF_TOKEN` — Enables split-screen / camera-switch mode ([get one here](https://huggingface.co/settings/tokens))

In [ ]:
# ──────────────────────────────────────────────────────────────────────
# Load API keys from Colab Secrets and create the .env file
# ──────────────────────────────────────────────────────────────────────
from google.colab import userdata
from pathlib import Path

# Required
GOOGLE_API_KEY = userdata.get("GOOGLE_API_KEY") or ""

# Optional — leave blank if you don't have them
PEXELS_API_KEY = userdata.get("PEXELS_API_KEY") or ""
HF_TOKEN = userdata.get("HF_TOKEN") or ""

# Write the .env file so the pipeline can read the keys
env_text = f"""# Auto-generated by Quick Start notebook
GOOGLE_API_KEY={GOOGLE_API_KEY}
PEXELS_API_KEY={PEXELS_API_KEY}
HF_TOKEN={HF_TOKEN}
"""
Path(".env").write_text(env_text, encoding="utf-8")

# Status check
print("🔑 API Key Status:")
print(f"   GOOGLE_API_KEY : {'✅ Set' if GOOGLE_API_KEY else '❌ Missing (required!)'}")
print(f"   PEXELS_API_KEY : {'✅ Set' if PEXELS_API_KEY else '⚪ Not set (optional)'}")
print(f"   HF_TOKEN       : {'✅ Set' if HF_TOKEN else '⚪ Not set (optional)'}")

if not GOOGLE_API_KEY:
    print("\n⚠️  WARNING: GOOGLE_API_KEY is missing!")
    print("   Add it to Colab Secrets (🔑 icon in sidebar) before running the pipeline.")
    print("   Get your free key at: https://aistudio.google.com/apikey")

## 4. Run the Pipeline 🎬

This cell runs the full AI clipping pipeline:

1. **Download** — Downloads the YouTube video
2. **Transcribe** — Converts speech to text using Faster-Whisper
3. **Analyze** — Google Gemini AI picks the most engaging moments
4. **Render** — Generates clips with face-tracking, subtitles, and hook teasers

> ⏱️ **Estimated time**: 5–15 minutes depending on video length and number of clips.

> ⚠️ Make sure you have set the `YOUTUBE_URL` in the Configuration section above!

In [ ]:
# ──────────────────────────────────────────────────────────────────────
# YouTube Cookies Setup (Optional) - Solves yt-dlp 403 Forbidden errors
# ──────────────────────────────────────────────────────────────────────
import os
import shutil

COOKIES_PATH = os.path.abspath("youtube_cookies.txt")
cookie_candidates = [
    "/kaggle/input/datasets/muhammadnaufal/tiktok-secret-cookies/youtube_cookies.txt",
    "/content/youtube_cookies.txt",
    COOKIES_PATH,
]

source_cookie = next((p for p in cookie_candidates if os.path.isfile(p)), None)
YT_COOKIES_ARG = ""

if source_cookie:
    if os.path.realpath(source_cookie) != os.path.realpath(COOKIES_PATH):
        shutil.copy(source_cookie, COOKIES_PATH)
    print(f"✅ YouTube cookies aktif: {COOKIES_PATH}")
    YT_COOKIES_ARG = f"--yt-cookies {COOKIES_PATH}"
else:
    print("ℹ️ youtube_cookies.txt tidak ditemukan. Download akan dicoba tanpa cookies.")


In [ ]:
# ──────────────────────────────────────────────────────────────────────
# Execute the OpenSource Clipping pipeline with your configuration
# ──────────────────────────────────────────────────────────────────────

!python main.py \
  --url "{YOUTUBE_URL}" \
  --clips {TOTAL_CLIPS} \
  --ratio "{ASPECT_RATIO}" \
  --font-style "{FONT_STYLE}" \
  --hook-duration {HOOK_DURATION} \
  --words-per-sub {WORDS_PER_SUBTITLE} \
  --whisper-model "{WHISPER_MODEL}" \
  --whisper-device "{WHISPER_DEVICE}" \
  --whisper-compute-type "{WHISPER_COMPUTE_TYPE}" \
  {YT_COOKIES_ARG}

## 5. View & Download Results

After the pipeline completes, your clips are saved in the `outputs/` folder.
Run the cells below to preview and download them.

In [ ]:
# ──────────────────────────────────────────────────────────────────────
# List all generated output files
# ──────────────────────────────────────────────────────────────────────
import os

outputs_dir = "outputs"
if os.path.exists(outputs_dir):
    files = os.listdir(outputs_dir)
    video_files = [f for f in files if f.endswith(".mp4")]
    print(f"🎬 Generated {len(video_files)} clip(s):\n")
    for f in sorted(files):
        size_mb = os.path.getsize(os.path.join(outputs_dir, f)) / (1024 * 1024)
        icon = "🎥" if f.endswith(".mp4") else "🖼️" if f.endswith((".jpg", ".png")) else "📄"
        print(f"   {icon} {f} ({size_mb:.1f} MB)")
else:
    print("❌ No outputs found. Make sure the pipeline ran successfully.")

In [ ]:
# ──────────────────────────────────────────────────────────────────────
# Download all output files as a ZIP archive
# ──────────────────────────────────────────────────────────────────────
import shutil
from google.colab import files

if os.path.exists(outputs_dir) and os.listdir(outputs_dir):
    zip_name = "clipping_outputs"
    shutil.make_archive(zip_name, "zip", outputs_dir)
    print(f"📦 Created {zip_name}.zip — downloading now...")
    files.download(f"{zip_name}.zip")
else:
    print("❌ No outputs to download.")

---

## 📖 Troubleshooting

| Problem | Solution |
|---------|----------|
| `GOOGLE_API_KEY not found` | Add your API key to Colab Secrets (🔑 sidebar) |
| `CUDA out of memory` | Use a smaller Whisper model (`small` or `medium`) |
| `float16 not supported` | Change `WHISPER_COMPUTE_TYPE` to `float32` |
| Pipeline takes too long | Reduce `TOTAL_CLIPS` to `1` for testing |
| No B-roll footage | Add `PEXELS_API_KEY` to Colab Secrets |

## 🔗 Links

- **GitHub**: [NaufalRizqullah/opensource-clipping](https://github.com/NaufalRizqullah/opensource-clipping)
- **Full Documentation**: See the [README](https://github.com/NaufalRizqullah/opensource-clipping/blob/main/README.md) for all CLI options
- **Get Gemini API Key**: [aistudio.google.com/apikey](https://aistudio.google.com/apikey)
- **Get Pexels API Key**: [pexels.com/api](https://www.pexels.com/api/)